In [ ]:
import pandas as pd
import numpy as np

# Load your enhanced Excel file
df = pd.read_excel("kse30_daily_data_engineered.xlsx")

# Clean column names for convenience
df.columns = df.columns.str.replace(" ", "_").str.upper()

# Convert date into proper datetime
df["DATE"] = pd.to_datetime(df["DATE"])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
def data_quality_report(df):
    report = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes,
        'Total Rows': len(df),
        'Null Values': df.isnull().sum().values,
        'Null %': (df.isnull().sum().values / len(df)) * 100,
        'Zero Values': (df == 0).sum().values,
        'Zero %': ((df == 0).sum().values / len(df)) * 100
    })
    
    # Optional: Filter to show only columns with issues
    return report.sort_values(by='Null %', ascending=False)

quality_df = data_quality_report(df)

In [ ]:
quality_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualizing null values
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Data Heatmap")
plt.show()

In [ ]:
# Drop rows where DELTA_WT is NaN
df_changed = df.dropna(subset=["DELTA_WT"])
quality_df = data_quality_report(df_changed)

In [ ]:
quality_df

In [ ]:
y = df_changed["FLOW_PROXY"]
features = [
    "DELTA_WT",
    "DELTA_PRICE",
    "DELTA_VOLUME",
    "PREV_IDX_WT",
    "PREV_FF_MCAP",
    "RETURN",
]

X = df_changed[features]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

preds = model.predict(X_test)

print("R²:", r2_score(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_
plt.barh(features, importances)
plt.title("Feature Importance")
plt.show()


In [ ]:
plt.plot(y_test.values, label="Actual")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("Predicted vs Actual Flows")
plt.show()


In [ ]:
df_changed["FLOW_CLASS"] = np.where(df_changed["FLOW_PROXY"] > 0, 1,
                     np.where(df_changed["FLOW_PROXY"] < 0, -1, 0))

In [ ]:
print(df_changed['FLOW_CLASS'])

In [ ]:
df_changed["RET_5"] = df_changed.groupby("SYMBOL")["PRICE"].pct_change(5)
df_changed["RET_20"] = df_changed.groupby("SYMBOL")["PRICE"].pct_change(20)

df_changed["VOLAT_5"] = df_changed.groupby("SYMBOL")["PRICE"].rolling(5).std().reset_index(0,drop=True)
df_changed["VOLAT_20"] = df_changed.groupby("SYMBOL")["PRICE"].rolling(20).std().reset_index(0,drop=True)

df_changed["VOL_SHOCK_5"] = df_changed.groupby("SYMBOL")["VOLUME"].pct_change(5)

df_changed["ABS_DELTA_WT"] = df_changed["DELTA_WT"].abs()

In [ ]:
rebalancing_dates = [
    "2020-03-15",
    "2020-09-15",
    "2021-03-15",
    "2021-09-15",
    "2022-03-15",
    "2022-09-15",
    "2023-03-15",
    "2023-09-15",
    "2024-03-15",
    "2024-09-15",
    "2025-03-15",
    "2025-09-15",
]
df_changed["DAYS_TO_REBAL"] = df_changed["DATE"].apply(
    lambda x: min([abs((x - pd.Timestamp(d)).days) for d in rebalancing_dates])
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

preds = model.predict(X_test)

print("R²:", r2_score(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))

In [ ]:
plt.plot(y_test.values, label="Actual")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("Predicted vs Actual Flows")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_
plt.barh(features, importances)
plt.title("Feature Importance")
plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
import statsmodels.api as sm

X2 = sm.add_constant(df_changed["FLOW_PROXY"])
y2 = df_changed["RETURN"].shift(-1)

model2 = sm.OLS(y2, X2, missing="drop").fit()
print(model2.summary())